# TickDB -> Spark Directo (SIN KAFKA)

Este notebook consume datos directamente desde el socket TCP del producer de TickDB,
sin pasar por Kafka. Ideal cuando:
- No necesitas replay de eventos
- Solo un consumidor
- Quieres menor latencia
- Simplicidad es prioridad

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import time

In [ ]:
spark = SparkSession.builder \
    .appName("TickDBDirecto") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark: {spark.version}")

## Esquema del evento

In [ ]:
event_schema = StructType([
    StructField("symbol", StringType(), True),
    StructField("last_price", DoubleType(), True),
    StructField("volume_24h", DoubleType(), True),
    StructField("high_24h", DoubleType(), True),
    StructField("low_24h", DoubleType(), True),
    StructField("price_change_24h", DoubleType(), True),
    StructField("price_change_percent_24h", DoubleType(), True),
    StructField("timestamp", LongType(), True),
    StructField("event_timestamp", LongType(), True),
    StructField("market", StringType(), True)
])

## Leer stream desde socket TCP

In [ ]:
df_raw = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", 9999) \
    .load()

print("Lectura de socket iniciada")

## Parsear JSON

In [ ]:
from pyspark.sql.functions import from_json

df_parsed = df_raw.select(
    from_json(col("value"), event_schema).alias("data")
).select("data.*")

print("Schema aplicado")

## Calcular latencia y métricas

In [ ]:
df_metrics = df_parsed.withColumn(
    "latency_ms", 
    col("event_timestamp") - col("timestamp")
).withColumn(
    "processing_time",
    current_timestamp()
)

## Filtrar válidos

In [ ]:
df_valid = df_metrics.filter(
    col("symbol").isNotNull() & 
    col("last_price").isNotNull()
)

## Salida a consola

In [ ]:
query = df_valid \
    .writeStream \
    .format("console") \
    .option("truncate", "true") \
    .trigger(processingTime="5 seconds") \
    .start()

print("Stream iniciado - mostrando datos en consola")

## Métricas después de 30 segundos

In [ ]:
time.sleep(30)

if query.lastProgress:
    p = query.lastProgress
    print("=== MÉTRICAS ===")
    print(f"Eventos entrada: {p.get('numInputRows', 0)}")
    print(f"Entrada evt/s: {p.get('inputRowsPerSecond', 0):.2f}")
    print(f"Proceso evt/s: {p.get('processedRowsPerSecond', 0):.2f}")